# 02 · Reference: stabilizer-qec

**What this notebook is for:** to show what the `stabilizer-qec` package gives us, which
parts we use, and how they line up with the papers.

**Source:** `stabilizer-qec` 2.1.0, https://pypi.org/project/stabilizer-qec/ ,
documentation: https://docs.rs/crate/stabilizer_qec/latest/source/python/README.md .
A Rust library with Python bindings. Its decoders are tested against PyMatching, ldpc and
beliefmatching (per its README).

| What | API | Paper |
|---|---|---|
| The gross code | `BivariateBicycleCode("gross")` | Bravyi et al., arXiv:2308.07915 |
| Check matrices, logicals | `.check_matrices()`, `.logicals()` | same |
| Shift automorphisms | `.automorphisms()` | Bravyi et al.; Tour de gross arXiv:2506.03094 |
| Memory circuit | `.memory_circuit(cycles, p, basis)` | Bravyi et al. depth-8 cycle + noise model |
| Logical measurement | `.logical_measurement_circuit(op, basis, pre, merged, post, p)` | Williamson & Yoder arXiv:2410.02213; Cross et al. arXiv:2407.18393 |
| Decoders | `BpOsd`, `RelayBp`, `AmbiguityClustering`, `Matching`, ... | Roffe et al. arXiv:2005.07016; Müller et al. arXiv:2506.01779; Wolanski & Barber arXiv:2406.14527 |

In [1]:
# Setup: make the gross_check folder importable (the .py files live one level up).
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "independence"))
print("gross_check folder:", ROOT)

gross_check folder: /home/claude/gross_check


## 1. The code object

In [2]:
import numpy as np, stabilizer_qec as sq
g = sq.BivariateBicycleCode("gross")
print("n =", g.n, " k =", g.k)
l, m, a, b = g.polynomials            # a property: (l, m, A monomials, B monomials)
print("torus l x m =", l, "x", m)
print("A monomials (x-power, y-power):", a, "  -> A = x^3 + y + y^2")
print("B monomials (x-power, y-power):", b, "  -> B = y^3 + x + x^2")

n = 144  k = 12
torus l x m = 12 x 6
A monomials (x-power, y-power): [(3, 0), (0, 1), (0, 2)]   -> A = x^3 + y + y^2
B monomials (x-power, y-power): [(0, 3), (1, 0), (2, 0)]   -> B = y^3 + x + x^2


In [3]:
HX, HZ = g.check_matrices()
print("H_X:", HX.shape, " H_Z:", HZ.shape)
print("weight of every check (row sums):", set(HX.sum(1)), set(HZ.sum(1)))
print("every qubit is in how many X checks / Z checks:", set(HX.sum(0)), set(HZ.sum(0)))
print("CSS condition H_X H_Z^T = 0 (mod 2):", not ((HX @ HZ.T) % 2).any())
LX, LZ = g.logicals()
print("logicals:", LX.shape, LZ.shape, "| pairing X_i Z_j anticommute iff i=j:",
      np.array_equal((LX @ LZ.T) % 2, np.eye(12, dtype=int)))

H_X: (72, 144)  H_Z: (72, 144)
weight of every check (row sums): {np.uint64(6)} {np.uint64(6)}
every qubit is in how many X checks / Z checks: {np.uint64(3)} {np.uint64(3)}
CSS condition H_X H_Z^T = 0 (mod 2): True
logicals: (12, 144) (12, 144) | pairing X_i Z_j anticommute iff i=j: True


## 2. Shift automorphisms

Shifting every qubit by x^a y^b on the torus maps the code to itself (12 × 6 = 72 shifts), and
each can be combined with the ZX-duality: 144 automorphisms in total. Each one acts on the 12
logical qubits as a CNOT-type circuit; `action` is that map as a 24 × 24 binary matrix
(12 X logicals then 12 Z logicals). Tour de gross uses these to move logical qubits into the
two slots (1 and 7) that can be measured directly.

stabilizer-qec gives the **logical action** only, not a noisy circuit for performing a shift.

In [4]:
auts = g.automorphisms()
print(len(auts), "automorphisms")
a1 = [a for a in auts if a.shift == (1, 0) and not a.dual][0]
print("shift x^1 y^0, no duality:")
print(a1.action)
print("is it a permutation of logicals (one 1 per row)?", bool((a1.action.sum(1) == 1).all()))

144 automorphisms
shift x^1 y^0, no duality:
[[0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

## 3. Circuits

In [5]:
import stim
mem = stim.Circuit(str(g.memory_circuit(12, 0.004)))
print("memory: qubits", mem.num_qubits, "detectors", mem.num_detectors, "observables", mem.num_observables)
meas = stim.Circuit(str(g.logical_measurement_circuit("f", "x", pre=1, merged=12, post=1, p=0.004)))
print("logical X(f,0) measurement: qubits", meas.num_qubits, "(extra = gauging ancillas)",
      "detectors", meas.num_detectors, "observables", meas.num_observables, "(0 = outcome, 1-12 = logical X's)")

memory: qubits 288 detectors 936 observables 12
logical X(f,0) measurement: qubits 325 (extra = gauging ancillas) detectors 1212 observables 13 (0 = outcome, 1-12 = logical X's)


## 4. Decoders: same circuit, three decoders

All three take the *undecomposed* detector error model (gross code faults can flip 3+ detectors).
- `BpOsd`: BP + ordered-statistics decoding (same as ldpc's; we use IBM's settings).
- `RelayBp`: IBM's Relay-BP.
- `AmbiguityClustering`: Riverlane's AC decoder (Wolanski & Barber); its docs say it beats
  BP-OSD order 7-8 and is up to 100× faster on these codes.

In [6]:
import time
SHOTS = 500
dem = mem.detector_error_model(decompose_errors=False)
dets, obs = mem.compile_detector_sampler(seed=1).sample(SHOTS, separate_observables=True)
for name, dec in [("BpOsd (IBM settings)", sq.BpOsd(dem, max_iter=80, osd_method="osd_cs", osd_order=10)),
                  ("RelayBp", sq.RelayBp(dem)),
                  ("AmbiguityClustering", sq.AmbiguityClustering(dem))]:
    t = time.time()
    f = int((np.asarray(dec.decode_batch(dets), bool) != obs).any(axis=1).sum())
    print(f"{name:22s} {f}/{SHOTS} = {f/SHOTS:.3f}  ({time.time()-t:.1f} s)")

BpOsd (IBM settings)   15/500 = 0.030  (18.5 s)


RelayBp                7/500 = 0.014  (81.6 s)


AmbiguityClustering    8/500 = 0.016  (5.6 s)


In [7]:
print("GPU available to stabilizer-qec's samplers:", sq.gpu_device())

GPU available to stabilizer-qec's samplers: None
